Detects COVID-19 through the usage of custom Convolutional Neural Network (CNN), the CNN employs transfer learning with pre-trained models such as the ResNet series, the DenseNet series and the VGG series. Data augmentation was also used to improve the model robustness. The best results were found with the pre-trained model DenseNet169. The results were later compiled into a conference research paper titled "Enhancing Automated COVID Diagnosis from Chest X-rays using Convolutional Neural Networks and Transfer Learning", the paper can be found in the following link "https://ieeexplore.ieee.org/document/10473272/authors#authors".

In [ ]:

!wget http://cb.lk/covid_19


In [ ]:
!unzip covid_19


Archive:  covid_19
replace CovidDataset/Val/Covid/88de9d8c39e946abd495b37cd07d89e5-6531-0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
Train_PATH = "CovidDataset/Train"
Val_PATH = "CovidDataset/Test"
# from fastai.vision.data import ImageDataLoaders
# from fastai.vision import *
# data = ImageDataBunch.from_folder(Train_PATH, train=Train_PATH, valid_pct=0.2, size=224, num_workers=4)
!pip install image-classifiers==0.2.2
!pip install image-classifiers==1.0.0b1
!pip install git+https://github.com/qubvel/classification_models.git

  Using cached image_classifiers-0.2.2-py2.py3-none-any.whl (72 kB)
  Attempting uninstall: image-classifiers
    Found existing installation: image-classifiers 1.0.0
    Uninstalling image-classifiers-1.0.0:
      Successfully uninstalled image-classifiers-1.0.0
  Using cached image_classifiers-1.0.0b1-py3-none-any.whl
  Attempting uninstall: image-classifiers
    Found existing installation: image-classifiers 0.2.2
    Uninstalling image-classifiers-0.2.2:
      Successfully uninstalled image-classifiers-0.2.2
  Cloning https://github.com/qubvel/classification_models.git to /tmp/pip-req-build-jh6imddl
  Running command git clone --filter=blob:none --quiet https://github.com/qubvel/classification_models.git /tmp/pip-req-build-jh6imddl
  Resolved https://github.com/qubvel/classification_models.git to commit a0f006e05485a34ccf871c421279864b0ccd220b
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
  Created wheel for image-classifiers:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
# import classification_models
import numpy as np
import cv2

import PIL.Image as Image
import os

import matplotlib.pylab as plt

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.applications import ResNet50
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from keras.layers import *
from keras.models import *
from keras.preprocessing import image
# from keras_applications.resnet import resnet34
import tensorflow
from tensorflow.keras.applications import VGG16, VGG19
from classification_models.keras import Classifiers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.densenet import DenseNet201,  DenseNet169, DenseNet121,  preprocess_input, decode_predictions
from tensorflow.keras.applications.resnet import ResNet50, ResNet101, ResNet152


class MetricsCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs['val_accuracy'] > 0.984:
            print("Validation accuracy reached more than 98.33%. Stopping training.")
            self.model.stop_training = True
        val_precision = logs.get('val_precision')
        val_specificity = logs.get('val_specificity_at_sensitivity')
        val_recall = logs.get('val_recall')
        val_auc = logs.get('val_auc')

        val_precision_str = f'{val_precision:.4f}' if val_precision is not None else 'N/A'
        val_specificity_str = f'{val_specificity:.4f}' if val_specificity is not None else 'N/A'
        val_recall_str = f'{val_recall:.4f}' if val_recall is not None else 'N/A'
        val_auc_str = f'{val_auc:.4f}' if val_auc is not None else 'N/A'

        print(f'Precision: {val_precision_str}, Specificity: {val_specificity_str}, Recall: {val_recall_str}, AUC: {val_auc_str}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
# import classification_models
import numpy as np
import cv2

import PIL.Image as Image
import os

import matplotlib.pylab as plt

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.applications import ResNet50
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from keras.layers import *
from keras.models import *
from keras.preprocessing import image
# from keras_applications.resnet import resnet34
import tensorflow
from tensorflow.keras.applications import VGG16, VGG19
from classification_models.keras import Classifiers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.densenet import preprocess_input, decode_predictions
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.densenet import DenseNet201, DenseNet121,  preprocess_input, decode_predictions
from tensorflow.keras.applications.resnet import ResNet50, ResNet101, ResNet152
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.metrics import Precision, SpecificityAtSensitivity, Recall, AUC
from tensorflow.keras.applications.densenet import DenseNet169, preprocess_input, decode_predictions
# for tensorflow.keras
# from classification_models.tfkeras import Classifiers
from tensorflow.keras.applications import VGG16, VGG19
tf.random.set_seed(42)
precision = Precision()
specificity = SpecificityAtSensitivity(0.5)
recall = Recall()
auc = AUC()
densenetmodel = ResNet101(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the pre-trained weights
for layer in densenetmodel.layers:
    layer.trainable = False



x = densenetmodel.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(512, activation ='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation ='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(128, activation ='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.25)(x)

x = Dense(1, activation ='sigmoid')(x)
model = Model(densenetmodel.input, x)

model.compile(loss=keras.losses.binary_crossentropy, optimizer=Adam(lr=0.01 ), metrics=['accuracy', precision, specificity, recall, auc])



In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_1[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                              

In [ ]:
# as you go deepter into the network you see more complex features
# example seeing a phone which is in your face vs one a meter away
# we start with 32 filters then 64 and finally 128
# the number of distinct patterns or (receptive field)  will be less at the lower layers
# the later layers will contain more complex patterns

In [ ]:
# train from scratch
noise_level = 0.1
train_datagen = image.ImageDataGenerator(


    rescale = 1./255,
    shear_range = 0.2,
    zoom_range = 0.15,
    horizontal_flip = True, # can't do vertical flip as it'll completely change the image

    width_shift_range=0.2,
    height_shift_range=0.2,

    rotation_range=20,
    brightness_range=[0.5, 1.5],

    channel_shift_range=20,
    preprocessing_function=lambda x: x + tf.random.normal(tf.shape(x), mean=0.0, stddev=noise_level, dtype=tf.float32)
)

test_dataset = image.ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    'CovidDataset/Train',
    target_size = (224, 224),
    # color_mode='rgb',
    batch_size = 8,
    class_mode = 'binary',
)


Found 224 images belonging to 2 classes.


In [ ]:
train_generator.class_indices
# covid and normal class
# covid catgeory is 0
# normal category is 1



{'Covid': 0, 'Normal': 1}

In [ ]:
validation_generator = test_dataset.flow_from_directory(
    'CovidDataset/Val',
    # color_mode='rgb',
    target_size = (224, 224),
    batch_size = 8,
    class_mode = 'binary',

)

Found 60 images belonging to 2 classes.


In [ ]:
# I have done a 80 :: 20 split
# 80% of the data to training
# 20% of it to validation


In [ ]:
hist = model.fit(
    train_generator,
    # steps_per_epoch=80,
    epochs=15,
    validation_data = validation_generator,
    callbacks=[MetricsCallback()]
    # validation_steps = 2

)



Epoch 1/15
28/28 [==============================] - 160s 5s/step - loss: 0.7757 - accuracy: 0.6250 - precision: 0.6373 - specificity_at_sensitivity: 0.7500 - recall: 0.5804 - auc: 0.6737 - val_loss: 0.6592 - val_accuracy: 0.5000 - val_precision: 0.5000 - val_specificity_at_sensitivity: 1.0000 - val_recall: 1.0000 - val_auc: 0.9794
Epoch 2/15
28/28 [==============================] - 125s 4s/step - loss: 0.5831 - accuracy: 0.7679 - precision: 0.7727 - specificity_at_sensitivity: 0.9018 - recall: 0.7589 - auc: 0.8067 - val_loss: 0.6429 - val_accuracy: 0.8833 - val_precision: 0.9259 - val_specificity_at_sensitivity: 0.9667 - val_recall: 0.8333 - val_auc: 0.9639
Epoch 3/15
28/28 [==============================] - 128s 5s/step - loss: 0.5241 - accuracy: 0.7991 - precision: 0.8131 - specificity_at_sensitivity: 0.9196 - recall: 0.7768 - auc: 0.8456 - val_loss: 0.6696 - val_accuracy: 0.5000 - val_precision: 0.0000e+00 - val_specificity_at_sensitivity: 0.9667 - val_recall: 0.0000e+00 - val_auc: 

In [ ]:
import cv2


# Load the image
img = cv2.imread('COVID-32.png')

# Preprocess the image
img = cv2.resize(img, (224, 224))
img = img.astype('float32') / 255.0
img = np.expand_dims(img, axis=0)

# Load the trained model

# Make a prediction on the image
prediction = model.predict(img)

print(prediction)
# Interpret the output
if prediction > 0.5:
    print('Covid +ve')
else:
    print('Covid -ve')



error: ignored